# Semi-Supervised Calibration Benchmarking

This notebook compares **unsupervised** (default) vs **semi-supervised** HMM
calibration for RNA modification detection.

### Approach

1. Run the full V1→V2→V3 pipeline with **unsupervised defaults** → baseline metrics
2. Generate training labels from known *E. coli* rRNA modification sites
3. Train a **semi-supervised** HMM (Platt-scaling calibrator + learned transitions)
4. Re-run V3 with trained parameters → calibrated metrics
5. Compare before/after with ROC, PR curves, and per-modification-type breakdown
6. Cross-validate (leave-one-contig-out) to check for overfitting

## 0. Setup & Paths

In [ ]:
import logging
import sys
import time
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
)
warnings.filterwarnings("ignore", category=RuntimeWarning)

TESTDATA = Path("../testdata").resolve()
assert TESTDATA.exists(), f"testdata directory not found: {TESTDATA}"

NATIVE_BAM   = TESTDATA / "native_0.bam"
NATIVE_FASTQ = TESTDATA / "native_0" / "fastq" / "pass.fq.gz"
NATIVE_BLOW5 = TESTDATA / "native_0" / "blow5" / "nanopore.blow5"
IVT_BAM      = TESTDATA / "control_0.bam"
IVT_FASTQ    = TESTDATA / "control_0" / "fastq" / "pass.fq.gz"
IVT_BLOW5    = TESTDATA / "control_0" / "blow5" / "nanopore.blow5"
REF_FASTA    = TESTDATA / "ref.fa"

for label, path in [
    ("native BAM",   NATIVE_BAM),
    ("native FASTQ", NATIVE_FASTQ),
    ("native BLOW5", NATIVE_BLOW5),
    ("IVT BAM",      IVT_BAM),
    ("IVT FASTQ",    IVT_FASTQ),
    ("IVT BLOW5",    IVT_BLOW5),
    ("ref FASTA",    REF_FASTA),
]:
    status = "\u2705" if path.exists() else "\u274c MISSING"
    print(f"  {status}  {label:15s}  {path}")

OUTPUT_DIR = Path("../output/semi_supervised").resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"\nOutput directory: {OUTPUT_DIR}")

## 1. Known Modifications & Labels

Known *E. coli* rRNA modification sites. Positions are **biological (1-based)**.
The mapping to pipeline positions is: `pipeline_pos + 3 = biological_pos`.

**Ground truth rule**:
- At known modification sites: **native reads → 1** (modified), **IVT reads → 0**
- At all other positions: **all reads → 0** (unmodified)

In [ ]:
POSITION_OFFSET = 3

KNOWN_MODIFICATIONS = {
    # --- Pseudouridine (Psi) ---
    ("ecoli16S", 516):  ("Psi", "pseudouridine"),
    ("ecoli23S", 746):  ("Psi", "pseudouridine"),
    ("ecoli23S", 955):  ("Psi", "pseudouridine"),
    ("ecoli23S", 1911): ("Psi", "pseudouridine"),
    ("ecoli23S", 1917): ("Psi", "pseudouridine"),
    ("ecoli23S", 2457): ("Psi", "pseudouridine"),
    ("ecoli23S", 2504): ("Psi", "pseudouridine"),
    ("ecoli23S", 2580): ("Psi", "pseudouridine"),
    ("ecoli23S", 2604): ("Psi", "pseudouridine"),
    ("ecoli23S", 2605): ("Psi", "pseudouridine"),
    # --- N2-methylguanosine (m2G) ---
    ("ecoli16S", 966):  ("m2G", "N2-methylguanosine"),
    ("ecoli16S", 1207): ("m2G", "N2-methylguanosine"),
    ("ecoli16S", 1516): ("m2G", "N2-methylguanosine"),
    ("ecoli23S", 1835): ("m2G", "N2-methylguanosine"),
    ("ecoli23S", 2445): ("m2G", "N2-methylguanosine"),
    # --- 5-methylcytidine (m5C) ---
    ("ecoli16S", 967):  ("m5C", "5-methylcytidine"),
    ("ecoli16S", 1407): ("m5C", "5-methylcytidine"),
    ("ecoli23S", 1962): ("m5C", "5-methylcytidine"),
    # --- 5-methyluridine (m5U) ---
    ("ecoli23S", 747):  ("m5U", "5-methyluridine"),
    ("ecoli23S", 1939): ("m5U", "5-methyluridine"),
    # --- N6,N6-dimethyladenosine (m66A) ---
    ("ecoli16S", 1518): ("m66A", "N6,N6-dimethyladenosine"),
    ("ecoli16S", 1519): ("m66A", "N6,N6-dimethyladenosine"),
    # --- N6-methyladenosine (m6A) ---
    ("ecoli23S", 1618): ("m6A", "N6-methyladenosine"),
    ("ecoli23S", 2030): ("m6A", "N6-methyladenosine"),
    # --- N7-methylguanosine (m7G) ---
    ("ecoli16S", 527):  ("m7G", "N7-methylguanosine"),
    ("ecoli23S", 2069): ("m7G", "N7-methylguanosine"),
    # --- Single-occurrence modifications ---
    ("ecoli23S", 2498): ("Cm",    "2\u2032-O-methyl cytosine"),
    ("ecoli23S", 2449): ("D",     "dihydrouridine"),
    ("ecoli23S", 2251): ("Gm",    "2\u2032-O-methyl guanine"),
    ("ecoli23S", 2552): ("Um",    "2\u2032-O-methyl uridine"),
    ("ecoli23S", 2501): ("ho5C",  "5-hydroxycytidine"),
    ("ecoli23S", 745):  ("m1G",   "1-methylguanosine"),
    ("ecoli23S", 2503): ("m2A",   "2-methyladenosine"),
    ("ecoli23S", 1915): ("m3Psi", "3-methylpseudouridine"),
    ("ecoli16S", 1498): ("m3U",   "3-methyluridine"),
    ("ecoli16S", 1402): ("m4Cm",  "N4,2\u2032-O-dimethylcytidine"),
}

KNOWN_MOD_PIPELINE = {
    (contig, bio_pos - POSITION_OFFSET): (mod_short, mod_full)
    for (contig, bio_pos), (mod_short, mod_full) in KNOWN_MODIFICATIONS.items()
}
KNOWN_MOD_PIPELINE_SET = set(KNOWN_MOD_PIPELINE.keys())

mod_counts = Counter(v[0] for v in KNOWN_MODIFICATIONS.values())
print(f"Total known modification sites: {len(KNOWN_MODIFICATIONS)}")
print(f"Modification types: {len(mod_counts)}")
print(f"Position offset: +{POSITION_OFFSET} (pipeline \u2192 biological)\n")
for mod_type, count in mod_counts.most_common():
    full = next(v[1] for v in KNOWN_MODIFICATIONS.values() if v[0] == mod_type)
    print(f"  {mod_type:<8s} {count:3d}  {full}")

## 2. Run Pipeline (f5c + DTW)

Run f5c eventalign once per contig and compute pairwise DTW distance matrices
using the best config from internal benchmarking: **padding=0, open_start=False,
open_end=False** (the default).

In [ ]:
from baleen.eventalign._bam import (
    get_contig_stats, filter_contigs, validate_bam,
    split_bam_contig,
)
from baleen.eventalign._f5c import check_f5c, index_fastq_blow5, index_blow5, run_eventalign
from baleen.eventalign._signal import (
    group_signals_by_position,
    get_common_positions,
    extract_signals_for_dtw,
)

MIN_DEPTH = 15

f5c_version = check_f5c()
print(f"f5c version: {f5c_version}")

print("Indexing...")
index_fastq_blow5(NATIVE_FASTQ, NATIVE_BLOW5)
index_fastq_blow5(IVT_FASTQ, IVT_BLOW5)
index_blow5(NATIVE_BLOW5)
index_blow5(IVT_BLOW5)

validate_bam(NATIVE_BAM)
validate_bam(IVT_BAM)
native_stats = get_contig_stats(NATIVE_BAM)
ivt_stats = get_contig_stats(IVT_BAM)
passed_contigs, filter_results = filter_contigs(native_stats, ivt_stats, min_depth=float(MIN_DEPTH))
print(f"Contigs passing filter: {passed_contigs}")

In [ ]:
import tempfile, shutil
from baleen.eventalign._pipeline import PositionResult, ContigResult
from baleen._cuda_dtw import dtw_pairwise_varlen, CUDA_AVAILABLE

print(f"CUDA available: {CUDA_AVAILABLE}")

contig_results: dict[str, ContigResult] = {}
tmp_root = Path(tempfile.mkdtemp(prefix="baleen-semisup-"))
print(f"Temp dir: {tmp_root}")

try:
    for contig in passed_contigs:
        print(f"\n{'='*50}")
        print(f"Processing contig: {contig}")
        contig_tmp = tmp_root / contig
        contig_tmp.mkdir(parents=True, exist_ok=True)

        native_contig_bam = split_bam_contig(NATIVE_BAM, contig, contig_tmp / "native")
        ivt_contig_bam = split_bam_contig(IVT_BAM, contig, contig_tmp / "ivt")

        native_tsv = contig_tmp / "native.eventalign.tsv"
        ivt_tsv = contig_tmp / "ivt.eventalign.tsv"

        print("  Running f5c eventalign (native)...")
        run_eventalign(native_contig_bam, REF_FASTA, NATIVE_FASTQ, NATIVE_BLOW5, native_tsv, rna=True)
        print("  Running f5c eventalign (IVT)...")
        run_eventalign(ivt_contig_bam, REF_FASTA, IVT_FASTQ, IVT_BLOW5, ivt_tsv, rna=True)

        native_by_pos = group_signals_by_position(native_tsv)
        ivt_by_pos = group_signals_by_position(ivt_tsv)
        common = get_common_positions(native_by_pos, ivt_by_pos)

        # Compute DTW with default config (padding=0, open_start=False, open_end=False)
        position_results: dict[int, PositionResult] = {}
        for pos in common:
            native_names, native_sigs = extract_signals_for_dtw(native_by_pos[pos])
            ivt_names, ivt_sigs = extract_signals_for_dtw(ivt_by_pos[pos])

            if not native_sigs or not ivt_sigs:
                continue

            valid_native = [(n, s) for n, s in zip(native_names, native_sigs) if len(s) > 0]
            valid_ivt = [(n, s) for n, s in zip(ivt_names, ivt_sigs) if len(s) > 0]
            if not valid_native or not valid_ivt:
                continue
            native_names = [n for n, _ in valid_native]
            native_sigs = [s for _, s in valid_native]
            ivt_names = [n for n, _ in valid_ivt]
            ivt_sigs = [s for _, s in valid_ivt]

            all_sigs = native_sigs + ivt_sigs
            matrix = dtw_pairwise_varlen(
                all_sigs,
                use_open_start=False,
                use_open_end=False,
                use_cuda=None,
            )

            kmer = native_by_pos[pos].reference_kmer
            position_results[pos] = PositionResult(
                position=pos,
                reference_kmer=kmer,
                n_native_reads=len(native_names),
                n_ivt_reads=len(ivt_names),
                native_read_names=native_names,
                ivt_read_names=ivt_names,
                distance_matrix=matrix,
            )

        nat_depth = native_stats[contig].mean_depth if contig in native_stats else 0.0
        ivt_depth = ivt_stats[contig].mean_depth if contig in ivt_stats else 0.0
        contig_results[contig] = ContigResult(
            contig=contig,
            native_depth=nat_depth,
            ivt_depth=ivt_depth,
            positions=position_results,
        )

        found_mods = [(contig, p) for p in common if (contig, p) in KNOWN_MOD_PIPELINE_SET]
        print(f"  \u2192 {len(position_results)} positions, {len(found_mods)} known mod sites")
        for c, p in found_mods:
            bio_pos = p + POSITION_OFFSET
            mod_short = KNOWN_MOD_PIPELINE[(c, p)][0]
            print(f"    pos {p} (bio {bio_pos}) \u2192 {mod_short}")
finally:
    shutil.rmtree(tmp_root, ignore_errors=True)
    print(f"\nCleaned up temp dir: {tmp_root}")

print(f"\nContigResults cached for {len(contig_results)} contigs.")

## 3. Baseline — Unsupervised Pipeline

Run `compute_sequential_modification_probabilities` with **no hmm_params**
(unsupervised defaults). Collect per-read binary labels and scores for both
V2 (`p_mod_raw`) and V3 (`p_mod_hmm`).

In [ ]:
from baleen.eventalign import compute_sequential_modification_probabilities


def compute_metrics(y_true: np.ndarray, y_score: np.ndarray) -> dict:
    """Compute AUROC, AUPRC, and F1 at optimal threshold."""
    if len(y_true) == 0 or y_true.sum() == 0 or y_true.sum() == len(y_true):
        return {"auroc": np.nan, "auprc": np.nan, "f1": np.nan,
                "precision": np.nan, "recall": np.nan, "threshold": np.nan,
                "n_total": len(y_true), "n_pos": int(y_true.sum()),
                "n_neg": int(len(y_true) - y_true.sum())}

    auroc = roc_auc_score(y_true, y_score)
    auprc = average_precision_score(y_true, y_score)

    precisions, recalls, thresholds = precision_recall_curve(y_true, y_score)
    f1s = np.where(
        (precisions[:-1] + recalls[:-1]) > 0,
        2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1]),
        0.0,
    )
    best_idx = np.argmax(f1s)

    return {
        "auroc": auroc, "auprc": auprc,
        "f1": float(f1s[best_idx]),
        "precision": float(precisions[:-1][best_idx]),
        "recall": float(recalls[:-1][best_idx]),
        "threshold": float(thresholds[best_idx]),
        "n_total": len(y_true), "n_pos": int(y_true.sum()),
        "n_neg": int(len(y_true) - y_true.sum()),
    }


def collect_predictions(
    v2_results: dict[str, "ContigModificationResult"],
    contig_results: dict[str, ContigResult],
    score_field: str = "p_mod_hmm",
) -> tuple[np.ndarray, np.ndarray]:
    """Collect (y_true, y_score) across all contigs from a given score field.

    Ground truth: at known mod positions, native=1, IVT=0; elsewhere all=0.
    """
    y_true_all, y_score_all = [], []

    for contig, cmr in v2_results.items():
        for pos, ps in cmr.position_stats.items():
            is_mod = (contig, pos) in KNOWN_MOD_PIPELINE_SET
            scores = getattr(ps, score_field)

            if is_mod:
                y_true_pos = np.concatenate([np.ones(ps.n_native), np.zeros(ps.n_ivt)])
            else:
                y_true_pos = np.zeros(ps.n_native + ps.n_ivt)

            y_true_all.append(y_true_pos)
            y_score_all.append(scores)

    yt = np.concatenate(y_true_all)
    ys = np.concatenate(y_score_all)
    valid = ~np.isnan(ys)
    return yt[valid], ys[valid]

In [ ]:
# Run unsupervised pipeline (no hmm_params)
print("Running unsupervised V1\u2192V2\u2192V3 pipeline...")
t0 = time.time()

baseline_results: dict[str, any] = {}
for contig, cr in contig_results.items():
    baseline_results[contig] = compute_sequential_modification_probabilities(cr)
    n_positions = len(baseline_results[contig].position_stats)
    print(f"  {contig}: {n_positions} positions")

print(f"Done in {time.time() - t0:.1f}s")

# Collect baseline metrics
yt_v2_base, ys_v2_base = collect_predictions(baseline_results, contig_results, "p_mod_raw")
yt_v3_base, ys_v3_base = collect_predictions(baseline_results, contig_results, "p_mod_hmm")

metrics_v2_base = compute_metrics(yt_v2_base, ys_v2_base)
metrics_v3_base = compute_metrics(yt_v3_base, ys_v3_base)

print(f"\nBaseline V2 (raw):  AUROC={metrics_v2_base['auroc']:.4f}  AUPRC={metrics_v2_base['auprc']:.4f}  F1={metrics_v2_base['f1']:.4f}")
print(f"Baseline V3 (HMM):  AUROC={metrics_v3_base['auroc']:.4f}  AUPRC={metrics_v3_base['auprc']:.4f}  F1={metrics_v3_base['f1']:.4f}")
print(f"Samples: {metrics_v3_base['n_total']} total ({metrics_v3_base['n_pos']}+ / {metrics_v3_base['n_neg']}-) ")

## 4. Generate Training Labels

Use `labels_from_known_modifications` to convert the known modification
positions into training labels. This automatically generates negative labels
from high-coverage positions not in the known set.

In [ ]:
from baleen.eventalign import labels_from_known_modifications

labels = labels_from_known_modifications(
    KNOWN_MODIFICATIONS,
    baseline_results,
    position_offset=POSITION_OFFSET,
    auto_negatives=True,
    min_coverage=5,
)

n_pos = sum(1 for v in labels.values() if v)
n_neg = sum(1 for v in labels.values() if not v)
print(f"Training labels: {len(labels)} total ({n_pos} positive, {n_neg} negative)")

# Per-contig breakdown
print("\nPer-contig breakdown:")
for contig in sorted(contig_results.keys()):
    c_pos = sum(1 for (c, _), v in labels.items() if c == contig and v)
    c_neg = sum(1 for (c, _), v in labels.items() if c == contig and not v)
    print(f"  {contig}: {c_pos} positive, {c_neg} negative")

## 5. Train Semi-Supervised HMM

Fit a Platt-scaling calibrator on V2 `p_mod_raw` scores at labeled positions,
learn transition parameters from read trajectories, and set `init_prob` from
the base rate of modifications.

In [ ]:
from baleen.eventalign import train_semi_supervised

print("Training semi-supervised HMM...")
t0 = time.time()

hmm_params = train_semi_supervised(
    baseline_results,
    labels,
    species_name="ecoli",
    learn_transitions=True,
)

print(f"Done in {time.time() - t0:.1f}s")
print(f"\nLearned HMM parameters:")
print(f"  Mode:             {hmm_params.mode}")
print(f"  p_stay_per_base:  {hmm_params.p_stay_per_base:.4f}")
print(f"  init_prob:        [{hmm_params.init_prob[0]:.4f}, {hmm_params.init_prob[1]:.4f}]")
print(f"  Training species: {hmm_params.training_species}")
print(f"  N training pos:   {hmm_params.n_training_positions}")
print(f"  N training reads: {hmm_params.n_training_reads}")

if hmm_params.emission_transform is not None:
    cal = hmm_params.emission_transform
    print(f"\n  Platt calibrator:")
    print(f"    a = {cal.a:.4f}")
    print(f"    b = {cal.b:.4f}")
    # Show what calibration does to some example scores
    test_scores = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
    calibrated = cal.transform(test_scores)
    print(f"\n  Calibration mapping:")
    for raw, cal_val in zip(test_scores, calibrated):
        print(f"    {raw:.2f} \u2192 {cal_val:.4f}")

## 6. Calibrated Pipeline

Re-run `compute_sequential_modification_probabilities` with the trained
`hmm_params`. V1 and V2 produce the same `p_mod_raw` — only V3 (HMM
smoothing) uses the calibrated emission transform and learned transitions.

In [ ]:
print("Running calibrated V1\u2192V2\u2192V3 pipeline with trained hmm_params...")
t0 = time.time()

calibrated_results: dict[str, any] = {}
for contig, cr in contig_results.items():
    calibrated_results[contig] = compute_sequential_modification_probabilities(
        cr, hmm_params=hmm_params
    )
    n_positions = len(calibrated_results[contig].position_stats)
    print(f"  {contig}: {n_positions} positions")

print(f"Done in {time.time() - t0:.1f}s")

# Collect calibrated metrics
yt_v3_cal, ys_v3_cal = collect_predictions(calibrated_results, contig_results, "p_mod_hmm")
metrics_v3_cal = compute_metrics(yt_v3_cal, ys_v3_cal)

print(f"\nCalibrated V3 (HMM):  AUROC={metrics_v3_cal['auroc']:.4f}  AUPRC={metrics_v3_cal['auprc']:.4f}  F1={metrics_v3_cal['f1']:.4f}")
print(f"Samples: {metrics_v3_cal['n_total']} total ({metrics_v3_cal['n_pos']}+ / {metrics_v3_cal['n_neg']}-) ")

## 7. Before vs After Comparison

In [ ]:
# Side-by-side table
comparison = pd.DataFrame({
    "Metric": ["AUROC", "AUPRC", "F1", "Precision", "Recall", "Threshold"],
    "Baseline V2 (raw)": [
        metrics_v2_base["auroc"], metrics_v2_base["auprc"], metrics_v2_base["f1"],
        metrics_v2_base["precision"], metrics_v2_base["recall"], metrics_v2_base["threshold"],
    ],
    "Baseline V3 (unsup)": [
        metrics_v3_base["auroc"], metrics_v3_base["auprc"], metrics_v3_base["f1"],
        metrics_v3_base["precision"], metrics_v3_base["recall"], metrics_v3_base["threshold"],
    ],
    "Calibrated V3 (semi-sup)": [
        metrics_v3_cal["auroc"], metrics_v3_cal["auprc"], metrics_v3_cal["f1"],
        metrics_v3_cal["precision"], metrics_v3_cal["recall"], metrics_v3_cal["threshold"],
    ],
})

# Add delta column
comparison["\u0394 (cal - unsup)"] = (
    comparison["Calibrated V3 (semi-sup)"] - comparison["Baseline V3 (unsup)"]
)

print("Before vs After Comparison")
print("=" * 80)
comparison.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors_plot = {"V2 raw": "#A0A0A0", "V3 unsupervised": "#5B9BD5", "V3 semi-supervised": "#ED7D31"}

# ROC curves
ax = axes[0]
for label, yt, ys, color in [
    ("V2 raw", yt_v2_base, ys_v2_base, colors_plot["V2 raw"]),
    ("V3 unsupervised", yt_v3_base, ys_v3_base, colors_plot["V3 unsupervised"]),
    ("V3 semi-supervised", yt_v3_cal, ys_v3_cal, colors_plot["V3 semi-supervised"]),
]:
    if len(yt) > 0 and yt.sum() > 0:
        fpr, tpr, _ = roc_curve(yt, ys)
        auroc_val = roc_auc_score(yt, ys)
        ax.plot(fpr, tpr, color=color, linewidth=2, label=f"{label} (AUROC={auroc_val:.3f})")

ax.plot([0, 1], [0, 1], "--", color="gray", linewidth=1)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves \u2014 Baseline vs Calibrated", fontweight="bold")
ax.legend(fontsize=9, loc="lower right")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)

# PR curves
ax = axes[1]
for label, yt, ys, color in [
    ("V2 raw", yt_v2_base, ys_v2_base, colors_plot["V2 raw"]),
    ("V3 unsupervised", yt_v3_base, ys_v3_base, colors_plot["V3 unsupervised"]),
    ("V3 semi-supervised", yt_v3_cal, ys_v3_cal, colors_plot["V3 semi-supervised"]),
]:
    if len(yt) > 0 and yt.sum() > 0:
        prec, rec, _ = precision_recall_curve(yt, ys)
        auprc_val = average_precision_score(yt, ys)
        ax.plot(rec, prec, color=color, linewidth=2, label=f"{label} (AUPRC={auprc_val:.3f})")

prevalence = metrics_v3_base["n_pos"] / max(metrics_v3_base["n_total"], 1)
ax.axhline(prevalence, color="gray", linestyle="--", linewidth=1, label=f"baseline ({prevalence:.3f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves \u2014 Baseline vs Calibrated", fontweight="bold")
ax.legend(fontsize=9, loc="upper right")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_pr_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Score distribution histograms: P(mod) at modified vs unmodified positions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
bins = np.linspace(0, 1, 50)

for col, (title, yt, ys) in enumerate([
    ("V3 Unsupervised (baseline)", yt_v3_base, ys_v3_base),
    ("V3 Semi-supervised (calibrated)", yt_v3_cal, ys_v3_cal),
]):
    mask_pos = yt == 1
    mask_neg = yt == 0

    # Top row: modified positions
    ax = axes[0][col]
    ax.hist(ys[mask_pos], bins=bins, alpha=0.7, color="#ED7D31", edgecolor="black", linewidth=0.3,
            label=f"Modified (n={mask_pos.sum()})")
    ax.set_title(f"{title}\nModified positions", fontweight="bold", fontsize=10)
    ax.set_xlabel("P(modified)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

    # Bottom row: unmodified positions
    ax = axes[1][col]
    ax.hist(ys[mask_neg], bins=bins, alpha=0.7, color="#5B9BD5", edgecolor="black", linewidth=0.3,
            label=f"Unmodified (n={mask_neg.sum()})")
    ax.set_title(f"{title}\nUnmodified positions", fontweight="bold", fontsize=10)
    ax.set_xlabel("P(modified)")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle("Score Distributions: Before vs After Calibration", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "score_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Cross-Validation

Leave-one-contig-out cross-validation to check that semi-supervised
calibration generalizes and doesn't overfit to the training contigs.

In [ ]:
from baleen.eventalign import cross_validate_hmm

print("Running leave-one-contig-out cross-validation...")
t0 = time.time()

try:
    cv_result = cross_validate_hmm(
        contig_results,
        labels,
        "semi_supervised",
        cv_strategy="leave_one_contig_out",
    )

    print(f"Done in {time.time() - t0:.1f}s")
    print(f"\nCross-validation results ({len(cv_result.per_fold_auroc)} folds):")
    print(f"  AUROC: {cv_result.mean_auroc:.4f} \u00b1 {cv_result.std_auroc:.4f}")
    print(f"  AUPRC: {cv_result.mean_auprc:.4f} \u00b1 {cv_result.std_auprc:.4f}")

    print("\nPer-fold details:")
    for fold in cv_result.fold_details:
        print(f"  Test={fold['test_contigs']}  AUROC={fold['auroc']:.4f}  "
              f"AUPRC={fold['auprc']:.4f}  n_reads={fold['n_test_reads']}")

    # Compare CV AUROC to full-data AUROC for overfitting check
    print(f"\nOverfitting check:")
    print(f"  Full-data AUROC:  {metrics_v3_cal['auroc']:.4f}")
    print(f"  CV mean AUROC:    {cv_result.mean_auroc:.4f}")
    gap = metrics_v3_cal['auroc'] - cv_result.mean_auroc
    print(f"  Gap:              {gap:.4f} {'(possible overfitting)' if gap > 0.05 else '(OK)'}")

except ValueError as e:
    print(f"Cross-validation skipped: {e}")
    cv_result = None

## 9. Per-Modification-Type Breakdown

Compare detection performance for each modification type (Psi, m5C, m6A, etc.)
between baseline (unsupervised) and calibrated (semi-supervised) V3.

In [ ]:
def collect_per_mod_type(
    results: dict[str, "ContigModificationResult"],
    score_field: str = "p_mod_hmm",
) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    """Collect (y_true, y_score) per modification type.

    Each mod type includes its own positive reads plus ALL negative reads
    (so AUROC is per-type, not just within-type).
    """
    # First collect all scores at unmodified positions
    neg_scores = []
    for contig, cmr in results.items():
        for pos, ps in cmr.position_stats.items():
            if (contig, pos) not in KNOWN_MOD_PIPELINE_SET:
                neg_scores.append(getattr(ps, score_field))

    neg_all = np.concatenate(neg_scores) if neg_scores else np.array([])
    neg_true = np.zeros(len(neg_all))

    # Per-type positives
    per_type: dict[str, tuple[list, list]] = {}
    for contig, cmr in results.items():
        for pos, ps in cmr.position_stats.items():
            if (contig, pos) not in KNOWN_MOD_PIPELINE_SET:
                continue
            mod_type = KNOWN_MOD_PIPELINE[(contig, pos)][0]
            if mod_type not in per_type:
                per_type[mod_type] = ([], [])

            scores = getattr(ps, score_field)
            # Native reads = positive, IVT reads = negative
            y_true_pos = np.concatenate([np.ones(ps.n_native), np.zeros(ps.n_ivt)])
            per_type[mod_type][0].append(y_true_pos)
            per_type[mod_type][1].append(scores)

    # Combine: each mod type gets its positives + all negatives
    result = {}
    for mod_type, (yt_list, ys_list) in per_type.items():
        yt = np.concatenate(yt_list + [neg_true])
        ys = np.concatenate(ys_list + [neg_all])
        valid = ~np.isnan(ys)
        result[mod_type] = (yt[valid], ys[valid])

    return result

In [ ]:
per_mod_base = collect_per_mod_type(baseline_results, "p_mod_hmm")
per_mod_cal = collect_per_mod_type(calibrated_results, "p_mod_hmm")

mod_types = sorted(set(per_mod_base.keys()) | set(per_mod_cal.keys()))

rows = []
for mt in mod_types:
    row = {"mod_type": mt}

    if mt in per_mod_base:
        m = compute_metrics(*per_mod_base[mt])
        row["base_auroc"] = m["auroc"]
        row["base_auprc"] = m["auprc"]
        row["n_pos"] = m["n_pos"]
    else:
        row["base_auroc"] = np.nan
        row["base_auprc"] = np.nan
        row["n_pos"] = 0

    if mt in per_mod_cal:
        m = compute_metrics(*per_mod_cal[mt])
        row["cal_auroc"] = m["auroc"]
        row["cal_auprc"] = m["auprc"]
    else:
        row["cal_auroc"] = np.nan
        row["cal_auprc"] = np.nan

    rows.append(row)

df_mod = pd.DataFrame(rows)
df_mod["\u0394_auroc"] = df_mod["cal_auroc"] - df_mod["base_auroc"]
df_mod["\u0394_auprc"] = df_mod["cal_auprc"] - df_mod["base_auprc"]
df_mod = df_mod.sort_values("base_auroc", ascending=False)

print("Per-modification-type comparison (baseline vs calibrated V3):")
df_mod.round(4)

In [ ]:
# Heatmap: mod_type x (baseline, calibrated) showing AUROC
if len(mod_types) > 0:
    heat_data = np.full((len(mod_types), 2), np.nan)
    for i, mt in enumerate(mod_types):
        row = df_mod[df_mod["mod_type"] == mt]
        if len(row) > 0:
            heat_data[i, 0] = row.iloc[0]["base_auroc"]
            heat_data[i, 1] = row.iloc[0]["cal_auroc"]

    fig, ax = plt.subplots(figsize=(6, max(4, 0.5 * len(mod_types))))
    im = ax.imshow(heat_data, aspect="auto", cmap="RdYlGn", vmin=0.4, vmax=1.0)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Baseline (unsup)", "Calibrated (semi-sup)"], fontsize=10)
    ax.set_yticks(range(len(mod_types)))
    ax.set_yticklabels(mod_types, fontsize=9)
    ax.set_title("AUROC per Modification Type\n(Baseline vs Calibrated)",
                 fontsize=12, fontweight="bold")

    for i in range(heat_data.shape[0]):
        for j in range(heat_data.shape[1]):
            if not np.isnan(heat_data[i, j]):
                ax.text(j, i, f"{heat_data[i,j]:.3f}", ha="center", va="center",
                        fontsize=9, color="black")

    fig.colorbar(im, ax=ax, label="AUROC", shrink=0.8)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "per_mod_type_heatmap.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No modification types found.")

## 10. Summary

In [ ]:
print("=" * 70)
print("SEMI-SUPERVISED CALIBRATION SUMMARY")
print("=" * 70)

print(f"\nDataset: E. coli rRNA ({len(contig_results)} contigs)")
print(f"Training labels: {n_pos} positive, {n_neg} negative positions")
print(f"Total reads: {metrics_v3_base['n_total']} ({metrics_v3_base['n_pos']}+ / {metrics_v3_base['n_neg']}-) ")

print(f"\nLearned parameters:")
print(f"  p_stay_per_base: {hmm_params.p_stay_per_base:.4f}  (default: 0.98)")
print(f"  init_prob:       [{hmm_params.init_prob[0]:.4f}, {hmm_params.init_prob[1]:.4f}]  (default: [0.5, 0.5])")
if hmm_params.emission_transform:
    print(f"  calibrator a:    {hmm_params.emission_transform.a:.4f}")
    print(f"  calibrator b:    {hmm_params.emission_transform.b:.4f}")

print(f"\n{'Metric':<15s} {'Baseline V3':>12s} {'Calibrated V3':>14s} {'Delta':>8s}")
print("-" * 55)
for metric_name, key in [("AUROC", "auroc"), ("AUPRC", "auprc"), ("F1", "f1"),
                          ("Precision", "precision"), ("Recall", "recall")]:
    base_val = metrics_v3_base[key]
    cal_val = metrics_v3_cal[key]
    delta = cal_val - base_val
    sign = "+" if delta >= 0 else ""
    print(f"  {metric_name:<13s} {base_val:>12.4f} {cal_val:>14.4f} {sign}{delta:>7.4f}")

if cv_result is not None:
    print(f"\nCross-validation ({len(cv_result.per_fold_auroc)} folds):")
    print(f"  CV AUROC: {cv_result.mean_auroc:.4f} \u00b1 {cv_result.std_auroc:.4f}")
    print(f"  CV AUPRC: {cv_result.mean_auprc:.4f} \u00b1 {cv_result.std_auprc:.4f}")

print("\n" + "=" * 70)

In [ ]:
# Save results
comparison.to_csv(OUTPUT_DIR / "calibration_comparison.csv", index=False)
print(f"Comparison table saved to: {OUTPUT_DIR / 'calibration_comparison.csv'}")

df_mod.to_csv(OUTPUT_DIR / "per_mod_type_comparison.csv", index=False)
print(f"Per-mod-type results saved to: {OUTPUT_DIR / 'per_mod_type_comparison.csv'}")

# Save trained HMM params
from baleen.eventalign import save_hmm_params
params_path = OUTPUT_DIR / "ecoli_semi_supervised_hmm.json"
save_hmm_params(hmm_params, params_path)
print(f"HMM params saved to: {params_path}")

---

**Done.** Results and trained HMM parameters are saved in the output directory.
Use `ecoli_semi_supervised_hmm.json` for cross-species transfer experiments.